In [1]:
#导入库
import duckdb
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import squarify 
import seaborn as sns
import squarify
from IPython.display import Image
# 画图的主题设计
sns.set_theme()
sns.set_context('notebook')

In [5]:
# ====================== 绘制矩形树图 =============================
def treemap(data, column, figsize=(12 ,8), choice = 0,title=None, ax=None):
    """
    data: 整份数据
    column：传入格式 'col_name'
    基准行，求改行各定类变量的个数，并以占比的情况反映在矩形区域中
    """ # 这个库需要 pip install 一下
    if choice:#这个画出来的商品图，标签包括商品id和类目id
        data1 = duckdb.sql("""select itemid,count(*) AS counts,max(categoryid) from data group by 1 order by count(*) desc,1 desc""").df()
        data = data.groupby(column).size().reset_index(name='counts')
        data = data.sort_values(by=['counts','itemid'],ascending=False)
        labels = data1.apply(lambda x: str(x[0]) + "\n (" + str(x[2]) + ")", axis=1)
    else: #这个画出来类目图
        data = data.groupby(column).size().reset_index(name='counts')
        data = data.sort_values(by=['counts',column],ascending=False)
        labels = data.apply(lambda x: str(x[0]) , axis=1)
    sizes = data['counts'].values.tolist() # 转成列表
    colors = [plt.cm.Spectral( i /float(len(labels))) for i in range(len(labels))]
    # Draw Plot
    plt.figure(figsize=figsize, dpi= 80) # 设置图片大小
    # 添加标签，图中部分元素
    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=.8, ax=ax)

    # Decorate
    plt.title(title)
    plt.axis('off') # 关闭坐标轴显示
    plt.title(title)
    plt.show()

从本地数据源建立sql数据表，加快读取速度（经测试较pandas直接读取快上两倍）以支持高达一亿的数据量读取

用sql语句对空值缺失值所在行进行删除处理（注：不用填充或是替换的方法是因为样本数据足够大而且填充的数据会一定程度上影响数据的真实性）

对用户id是空值的脏数据（日期处于11月25日和12月3日之外的数据）进行删除处理

In [4]:
start = time.time()
duckdb.sql("""
drop table if exists tmp_user_info
;
create table tmp_user_info
as
select * from read_csv("/Users/shenyunzhi/Desktop/淘宝天池项目/UserBehavior.csv",columns = {
        'userid': 'BIGINT',
        'itemid': 'BIGINT',
        'categoryid': 'BIGINT',
        'type': 'VARCHAR',
        'timestamp': 'BIGINT'
    },null_padding=true)
where strftime(to_timestamp(timestamp),'%Y-%m-%d') between '2017-11-25' and '2017-12-03';
DELETE FROM tmp_user_info
WHERE userid is null;
""")
end = time.time()
print(end - start)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

41.38230800628662


进行抽样

In [7]:
duckdb.sql("""
drop table if exists new_user_info
;
create table new_user_info
as
select * from tmp_user_info
where userid in (select userid from tmp_user_info group by 1 limit 9512)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

检验重复值

In [8]:
duckdb.sql("""
SELECT userid,itemid,timestamp FROM new_user_info
GROUP BY 1,2,3
HAVING COUNT(userid) > 1;
""")

┌────────┬────────┬───────────┐
│ userid │ itemid │ timestamp │
│ int64  │ int64  │   int64   │
├────────┴────────┴───────────┤
│           0 rows            │
└─────────────────────────────┘

检测无重复值  
数据预处理维度与数量分别如下：

In [9]:
duckdb.sql("""
(select count(distinct userid) AS "用户",
count(distinct itemid) AS "商品",
count(distinct categoryid) AS "商品类目",
count(distinct type) AS "行为",
count(distinct timestamp) AS "时间",
count(*) AS "数据量（条）" from new_user_info)
""")

┌───────┬────────┬──────────┬───────┬────────┬──────────────┐
│ 用户  │  商品  │ 商品类目 │ 行为  │  时间  │ 数据量（条） │
│ int64 │ int64  │  int64   │ int64 │ int64  │    int64     │
├───────┼────────┼──────────┼───────┼────────┼──────────────┤
│  9512 │ 391798 │     5790 │     4 │ 490284 │       965492 │
└───────┴────────┴──────────┴───────┴────────┴──────────────┘